# Gemini Human Recognize

In [1]:
%pip install --upgrade --quiet google-genai pillow dotenv pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [3]:
PROJECT_ID = os.environ['PROJECT_ID']
LOCATION = os.environ['LOCATION']
GS_BUCKET = os.environ['GS_BUCKET']
BUCKET = os.environ['BUCKET']

In [4]:
MODEL_ID= "gemini-2.0-flash-001"

In [46]:
from google import genai
from google.genai import types
from google.genai.types import (
    GenerateContentConfig,
    GoogleSearch,
    Part,
    Tool,
)
from IPython.display import HTML, Markdown, display
import pandas as pd

In [6]:
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

In [68]:
def detect_with_gemini(image_path):
    #system_instruction="You are an image analysis AI. You can identify people in images and provide their names"
    system_instruction="당신은 이미지 분석하는 일을 담당합니다. 주어진 이미지에서 인물을 찾아서 인물의 이름을 알려주세요. 인물이 누구인지 모르거나, 100% 신뢰할 수 없다면 '모름'으로 답하세요."

    google_search_tool = Tool(google_search=GoogleSearch())

    grounding_config = GenerateContentConfig(
        temperature = 0.3,
        top_p = 0.8,
        max_output_tokens = 8192,
        response_modalities = ["TEXT"],
        system_instruction=system_instruction,
        tools=[google_search_tool]   
    )

    #prompt = f"""If you can identify the people in the image, provide their names ONLY. If you don't know them, Don't make up names and just respond with 'unknown'"""
    prompt = f"""설명을 붙이지 말고, 이름만 대답하세요."""

    with open(image_path, "rb") as f:
        image = f.read()

    response = client.models.generate_content(
        model=MODEL_ID,
        contents=[Part.from_bytes(data=image, mime_type="image/png"),
            prompt,],
        config=grounding_config
    )

    print(response.text)


In [58]:
def detect_with_gemini_url(image_path, percent, grounding=True):
    #system_instruction="You are an image analysis AI. You can identify people in images and provide their names"
    system_instruction=f"당신은 이미지 분석하는 일을 담당합니다. 주어진 이미지에서 인물을 찾아서 인물의 이름을 알려주세요. 인물이 누구인지 모르거나, {percent}% 신뢰할 수 없다면 '모름'으로 답하세요."

    google_search_tool = Tool(google_search=GoogleSearch())

    grounding_config = GenerateContentConfig(
        temperature = 0.1,
        top_p = 0.99,
        max_output_tokens = 8192,
        response_modalities = ["TEXT"],
        system_instruction=system_instruction,
        tools=[google_search_tool]   
    )

    if grounding:
        grounding_config = GenerateContentConfig(
            temperature = 0.1,
            top_p = 0.99,
            max_output_tokens = 8192,
            response_modalities = ["TEXT"],
            system_instruction=system_instruction,
            tools=[google_search_tool]   
        )
    else:
        grounding_config = GenerateContentConfig(
            temperature = 0.1,
            top_p = 0.99,
            max_output_tokens = 8192,
            response_modalities = ["TEXT"],
            system_instruction=system_instruction,  
        )

    #prompt = f"""If you can identify the people in the image, provide their names ONLY. If you don't know them, Don't make up names and just respond with 'unknown'"""
    prompt = f"""설명을 붙이지 말고, 이름만 대답하세요. 
                output : 홍길동 """


    response = client.models.generate_content(
        model=MODEL_ID,
        contents=[Part.from_uri(file_uri=f"{GS_BUCKET}/human_images_4/{image_path}", mime_type="image/jpeg"),
            prompt,],
        config=grounding_config
    )

    print(response.text)
    return response.text

In [52]:
def evaluate(df):
    answer_df = pd.read_csv("./human_names.csv", index_col=False)
    df = df.sort_values(by="idx").reset_index(drop=True)
    df['answer'] = answer_df.iloc[:len(df), 0].values 
    df['name'] = df['name'].str.replace('\n', '', regex=False)

    comparison_result = df['name'] == df['answer']
    matching_count = comparison_result.sum()
    print(matching_count)
    total_count = len(df)
    unknown_count = (df['name'] == '모름').sum()

    print(f"Total :{total_count}, 모름 : {unknown_count}, 불일치 : {total_count - matching_count}, 모름 제외 불일치 : {total_count - matching_count - unknown_count}")
    return df

In [ ]:
def display_dataframe_with_images(df):

    html = df.copy()
    html = html.to_html(escape=False, index=False)

    display(HTML(html))

## with Grounding

In [ ]:
df = pd.DataFrame(columns=['idx','image', 'name'])

for i in range(1,101):
    name = detect_with_gemini_url(f"image_{i}.jpeg", 100)    
    image_data = f"<img src=\"https://storage.googleapis.com/{BUCKET}/human_images_4/image_{i}.jpeg\" width=\"40%\"/>"
    new_row = pd.DataFrame({'idx': [i],'image': [image_data], 'name': [name.replace('\n', '')]})
    df = pd.concat([df, new_row], ignore_index=True)


In [70]:
df_with_answer = evaluate(df)

36
Total :100, 모름 : 57, 불일치 : 64, 모름 제외 불일치 : 7


In [61]:
display_dataframe_with_images(df_with_answer)


idx,image,name,answer
1,,뷔,뷔
2,,모름,장윤정
3,,모름,설민석
4,,김태리,김태리
5,,김수현,김수현
6,,모름,차준환
7,,모름,홍석천
8,,모름,김지선
9,,모름,유관순
10,,아이유,아이유


## Without Grounding

In [ ]:
df = pd.DataFrame(columns=['idx','image', 'name'])

for i in range(1,101):
    name = detect_with_gemini_url(f"image_{i}.jpeg", 100, False)    
    image_data = f"<img src=\"https://storage.googleapis.com/{BUCKET}/human_images_4/image_{i}.jpeg\" width=\"40%\"/>"
    new_row = pd.DataFrame({'idx': [i],'image': [image_data], 'name': [name.replace('\n', '')]})
    df = pd.concat([df, new_row], ignore_index=True)


In [63]:
df_with_answer = evaluate(df)

43
Total :100, 모름 : 22, 불일치 : 57, 모름 제외 불일치 : 35


In [64]:
display_dataframe_with_images(df_with_answer)

idx,image,name,answer
1,,뷔,뷔
2,,정유미,장윤정
3,,전현무,설민석
4,,김태리,김태리
5,,김수현,김수현
6,,우노 쇼마,차준환
7,,모름,홍석천
8,,김신영,김지선
9,,박차정,유관순
10,,아이유,아이유
